In [ ]:
# import tools to work with numbers and files
# import tools to process satellite images and maps
# set folder locations for loading input data and saving output files
from pathlib import Path
import os
import warnings
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import from_bounds, xy as rio_xy
from rasterio.features import geometry_mask
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as mplcm
import matplotlib.colors as mplcolors
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.manifold import TSNE
from scipy.ndimage import gaussian_filter
from scipy.special import expit as sigmoid
from skimage.filters import gabor
from skimage.feature import structure_tensor, structure_tensor_eigenvalues
from skimage.morphology import skeletonize, remove_small_objects
import networkx as nx
from shapely.geometry import LineString, mapping
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 150, "axes.titlesize": 9, "axes.labelsize": 8})

load_dotenv()
try:
    _base = Path(__file__).resolve().parent.parent
except NameError:
    _base = Path.cwd().parent if Path.cwd().name == "spectral_analysis" else Path.cwd()

DATA_ENV = os.environ.get("SPECTRAL_DATA_DIR", "data/ElBagawat_SAR_Crops_Transportable")
DATA = Path(DATA_ENV) if Path(DATA_ENV).is_absolute() else _base / DATA_ENV

OUT_ENV = os.environ.get("SPECTRAL_OUT_DIR", "outputs/spectral")
OUT = Path(OUT_ENV) if Path(OUT_ENV).is_absolute() else _base / OUT_ENV
OUT.mkdir(parents=True, exist_ok=True)

OUT_SPEC = OUT
OUT_VEC = _base / "outputs" / "vector_gis"
OUT_VEC.mkdir(parents=True, exist_ok=True)

In [ ]:
# set names and wavelengths for the eight satellite color bands
# set colors and labels to draw charts and graphs
WV2_BANDS = ["Coastal", "Blue", "Green", "Yellow", "Red", "RedEdge", "NIR1", "NIR2"]
WAVELENGTHS = [425, 480, 545, 605, 660, 725, 833, 950]

ABSCAL = {
    "Coastal": 9.295654e-03, "Blue": 1.783568e-02, "Green": 1.364197e-02,
    "Yellow":  6.810718e-03, "Red":  1.851735e-02, "RedEdge": 6.063145e-03,
    "NIR1":    2.050828e-02, "NIR2": 9.042234e-03,
}

EFF_BW = {
    "Coastal": 4.73e-02, "Blue": 5.43e-02, "Green": 6.30e-02,
    "Yellow":  3.74e-02, "Red":  5.74e-02, "RedEdge": 3.93e-02,
    "NIR1":    9.89e-02, "NIR2": 9.96e-02,
}

In [ ]:
# create function to read map and image files from disk
# create function to save map and image files to disk
# create function to resize maps and match coordinate grids
# create function to scale values between zero and one for drawing
# create function to hide empty border areas around study site
def load_tif(path, nodata=-9999.0):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        meta = src.meta.copy()
        tf = src.transform
        crs = src.crs
    if nodata is not None:
        arr[arr == nodata] = np.nan
    return arr, meta, tf, crs

def save_tif(arr, path, tf, crs, nodata=None, dtype=None):
    if dtype is None:
        dtype = arr.dtype
    if nodata is None:
        nodata = -9999.0 if np.issubdtype(dtype, np.floating) else -1
    out = np.where(np.isnan(arr), nodata, arr) if np.issubdtype(arr.dtype, np.floating) else arr
    with rasterio.open(
        path, "w", driver="GTiff",
        height=out.shape[0], width=out.shape[1],
        count=1, dtype=dtype, crs=crs, transform=tf,
        nodata=nodata, compress="lzw",
    ) as dst:
        dst.write(out.astype(dtype), 1)
    print(f"saved: {path.name}")

def norm01(arr, pct=(2, 98)):
    valid = arr[~np.isnan(arr)]
    if valid.size == 0:
        return np.zeros_like(arr)
    lo, hi = np.percentile(valid, pct)
    return np.clip((arr - lo) / (hi - lo + 1e-9), 0, 1)

def safe_div(a, b, nodata_mask=None):
    with np.errstate(invalid="ignore", divide="ignore"):
        out = np.where(b != 0, a / b, 0.0)
    if nodata_mask is not None:
        out[nodata_mask] = np.nan
    return out.astype(np.float32)

def make_rgb(r_band, g_band, b_band, nodata_mask=None, pct=(2, 98)):
    r = norm01(r_band, pct)
    g = norm01(g_band, pct)
    b = norm01(b_band, pct)
    rgb = np.stack([r, g, b], axis=2)
    if nodata_mask is not None:
        rgb[nodata_mask] = 0
    return rgb

def spectral_angle(cube_flat, ref):
    dot = cube_flat @ ref
    norm_cube = np.linalg.norm(cube_flat, axis=1)
    cos_theta = dot / (norm_cube * np.linalg.norm(ref) + 1e-12)
    return np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))

def sample_raster_along_line(geom, raster_arr, raster_tf, step_m=2.0):
    if geom is None or geom.is_empty:
        return np.array([np.nan])
    length = geom.length
    distances = np.arange(0, max(length, step_m), step_m)
    pts = [geom.interpolate(d) for d in distances]
    H, W = raster_arr.shape
    vals = []
    for pt in pts:
        c_f, r_f = ~raster_tf * (pt.x, pt.y)
        r, c = int(r_f), int(c_f)
        if 0 <= r < H and 0 <= c < W:
            v = raster_arr[r, c]
            vals.append(float(v) if not np.isnan(v) else np.nan)
        else:
            vals.append(np.nan)
    return np.array(vals, dtype=np.float32)

def skeleton_to_geojson(skeleton, tf, crs):
    coords_set = set(zip(*np.where(skeleton)))
    if not coords_set:
        return gpd.GeoDataFrame(columns=["geometry", "n_pixels"], crs=crs)
    G = nx.Graph()
    moves = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
    for r, c in coords_set:
        G.add_node((r, c))
        for dr, dc in moves:
            nb = (r+dr, c+dc)
            if nb in coords_set:
                G.add_edge((r, c), nb)
    terminal = [n for n, d in G.degree() if d != 2]
    if not terminal:
        terminal = [next(iter(G.nodes))]
    features, visited = [], set()
    for start in terminal:
        for end in terminal:
            if start >= end:
                continue
            if not nx.has_path(G, start, end):
                continue
            try:
                path = nx.shortest_path(G, start, end)
            except nx.NetworkXNoPath:
                continue
            mid = path[1:-1]
            if not all(G.degree[n] == 2 for n in mid):
                continue
            sig = frozenset(tuple(sorted([path[i], path[i+1]])) for i in range(len(path)-1))
            if sig.issubset(visited):
                continue
            visited.update(sig)
            if len(path) >= 2:
                xy_coords = [rio_xy(tf, int(r), int(c)) for r, c in path]
                features.append({"geometry": LineString(xy_coords), "n_pixels": len(path)})
    if not features:
        return gpd.GeoDataFrame(columns=["geometry", "n_pixels"], crs=crs)
    return gpd.GeoDataFrame(features, crs=crs)

def max_gabor_responses(image, freq, n_angles=9):
    img = np.where(np.isnan(image), 0.0, image).astype(np.float64)
    angles = np.linspace(0, np.pi, n_angles, endpoint=False)
    best = np.zeros_like(img)
    for theta in angles:
        fr, fi = gabor(img, frequency=freq, theta=theta)
        mag = np.sqrt(fr**2 + fi**2)
        best = np.maximum(best, mag)
    best[np.isnan(image)] = np.nan
    return best.astype(np.float32)

In [ ]:
# set boundary coordinates for our study area around the ruins
# load and stitch together map tiles inside the boundary
# save cropped image of our study site
ROI_LEFT = 253_959
ROI_BOTTOM = 2_820_683
ROI_RIGHT = 254_435
ROI_TOP = 2_821_369

def get_mul_tiles(scene_dir):
    return sorted(scene_dir.glob("*M2AS_R*C*.TIF"))

def tile_overlaps_roi(tile_path, left, bottom, right, top):
    with rasterio.open(tile_path) as src:
        bnd = src.bounds
        return bnd.right > left and bnd.left < right and bnd.top > bottom and bnd.bottom < top

scene_dirs = [
    DATA / "DigitalGlobe_2018/MONO/058239078010_01/058239078010_01_P001_MUL",
    DATA / "DigitalGlobe_2018/MONO/058239078010_01/058239078010_01_P002_MUL",
    DATA / "DigitalGlobe_2018/MONO/058239078020_01/058239078020_01_P001_MUL",
]

all_tiles = []
for sd in scene_dirs:
    all_tiles.extend(get_mul_tiles(sd))

overlapping_tiles = [t for t in all_tiles if tile_overlaps_roi(t, ROI_LEFT, ROI_BOTTOM, ROI_RIGHT, ROI_TOP)]
print(f"Found {len(all_tiles)} total MUL tiles | {len(overlapping_tiles)} tiles overlap El Bagawat ROI")

tile_handles = [rasterio.open(t) for t in overlapping_tiles]
mosaic, mosaic_tf = merge(tile_handles, bounds=(ROI_LEFT, ROI_BOTTOM, ROI_RIGHT, ROI_TOP))

H, W = mosaic.shape[1], mosaic.shape[2]
crs = tile_handles[0].crs
print(f"Cropped mosaic shape: {mosaic.shape} ({H}×{W} px) | CRS: {crs}")
print(f"Memory footprint: ~{mosaic.nbytes / 1e6:.2f} MB")
for h in tile_handles:
    h.close()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("WV-2 ROI Mosaic — All 8 Bands (DN)", fontsize=11, fontweight="bold")
for i, (ax, name) in enumerate(zip(axes.flat, WV2_BANDS)):
    band = mosaic[i].astype(float)
    p2, p98 = np.percentile(band[band > 0], [2, 98])
    ax.imshow(band, vmin=p2, vmax=p98, cmap="gray", interpolation="nearest")
    ax.set_title(f"B{i+1} {name} ({WAVELENGTHS[i]}nm)")
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "01_raw_mosaic_all_bands.png", bbox_inches="tight")
plt.show()

In [ ]:
# convert raw satellite sensor numbers into real brightness levels
# load calibration numbers and calculate light intensity across the site
# save calibrated brightness map to file
mos_f = mosaic.astype(np.float32)
nodata_mask = mosaic[0] == 0

radiance = np.zeros_like(mos_f)
for i, name in enumerate(WV2_BANDS):
    radiance[i] = mos_f[i] * ABSCAL[name] / EFF_BW[name]

print("Radiance stats per band:")
for i, name in enumerate(WV2_BANDS):
    v = radiance[i][~nodata_mask]
    print(f"  {name:10s}: mean={v.mean():.4f}  std={v.std():.4f}  max={v.max():.4f}")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("WV-2 TOA Radiance (W/m²/sr/µm)", fontsize=11, fontweight="bold")
for i, (ax, name) in enumerate(zip(axes.flat, WV2_BANDS)):
    b = radiance[i].copy()
    b[nodata_mask] = np.nan
    p2, p98 = np.nanpercentile(b, [2, 98])
    im = ax.imshow(b, vmin=p2, vmax=p98, cmap="viridis", interpolation="nearest")
    ax.set_title(f"{name} ({WAVELENGTHS[i]}nm)")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT / "02_radiance_all_bands.png", bbox_inches="tight")
plt.show()

In [ ]:
# create normal picture using red green and blue light
# create infrared picture to highlight plants and ground types
# create geology picture to highlight rock features
# create shortwave picture to highlight soil and mineral differences
band_idx = {name: i for i, name in enumerate(WV2_BANDS)}

tc  = make_rgb(radiance[band_idx["Red"]],     radiance[band_idx["Green"]], radiance[band_idx["Blue"]], nodata_mask)
cir = make_rgb(radiance[band_idx["NIR1"]],    radiance[band_idx["Red"]],   radiance[band_idx["Green"]], nodata_mask)
geo = make_rgb(radiance[band_idx["RedEdge"]], radiance[band_idx["NIR1"]],  radiance[band_idx["Blue"]], nodata_mask)
sw  = make_rgb(radiance[band_idx["NIR2"]],    radiance[band_idx["RedEdge"]], radiance[band_idx["Green"]], nodata_mask)

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
fig.suptitle("WV-2 Color Composites — El Bagawat", fontsize=11, fontweight="bold")
for ax, img, title in zip(axes, [tc, cir, geo, sw], ["True Color (R-G-B)", "CIR (NIR1-R-G)", "Geology (RE-NIR1-B)", "NIR2-RE-G"]):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "03_color_composites.png", bbox_inches="tight")
plt.show()

In [ ]:
# calculate plant growth index to see where vegetation grows
# calculate water content index to find moisture
# calculate red edge index to measure subtle plant health
# calculate bare soil index to map open ground and sand
# calculate clay index to identify mineral composition
# calculate green vegetation index using green and infrared light
# calculate forest and desert index to separate bare rocks from plants
R  = radiance[band_idx["Red"]].astype(np.float64)
G  = radiance[band_idx["Green"]].astype(np.float64)
B  = radiance[band_idx["Blue"]].astype(np.float64)
RE = radiance[band_idx["RedEdge"]].astype(np.float64)
N1 = radiance[band_idx["NIR1"]].astype(np.float64)
N2 = radiance[band_idx["NIR2"]].astype(np.float64)

NDVI  = safe_div(N1 - R,  N1 + R, nodata_mask)
NDWI  = safe_div(G  - N1, G  + N1, nodata_mask)
NDRE  = safe_div(N1 - RE, N1 + RE, nodata_mask)
BSI   = safe_div((R + B) - (N1 + G), (R + B) + (N1 + G), nodata_mask)
CBI   = safe_div(N2 - RE, N2 + RE, nodata_mask)
GNDVI = safe_div(N1 - G,  N1 + G, nodata_mask)
FDI   = (RE - (R + (N1 - B) * 0.5)).astype(np.float32)
FDI[nodata_mask] = np.nan

indices = {
    "NDVI":  (NDVI,  "RdYlGn",   (-0.3,  0.3)),
    "NDWI":  (NDWI,  "Blues",    (-0.5,  0.3)),
    "NDRE":  (NDRE,  "PuOr",     (-0.5,  0.5)),
    "BSI":   (BSI,   "YlOrBr",   (-0.3,  0.5)),
    "CBI":   (CBI,   "cool",     (-0.4,  0.4)),
    "GNDVI": (GNDVI, "YlGn",     (-0.3,  0.5)),
    "FDI":   (FDI,   "Spectral_r",
              (float(np.nanpercentile(FDI[~nodata_mask], 5)),
               float(np.nanpercentile(FDI[~nodata_mask], 95)))),
}

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("WV-2 Spectral Indices — El Bagawat", fontsize=12, fontweight="bold")
for ax, (name, (idx, cmap, vrange)) in zip(axes.flat, indices.items()):
    im = ax.imshow(idx, cmap=cmap, vmin=vrange[0], vmax=vrange[1], interpolation="nearest")
    ax.set_title(name, fontsize=9)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
axes.flat[-1].axis("off")
plt.tight_layout()
plt.savefig(OUT / "04_spectral_indices.png", bbox_inches="tight")
plt.show()

In [ ]:
# prepare brightness values across all eight color bands
# find the main patterns of variation across the image
valid = ~nodata_mask

region_masks = {
    "High-BSI desert sand":      (BSI  > 0.2)  & valid,
    "Low-BSI structures":        (BSI  < 0.0)  & valid & (N1 > float(np.nanmedian(N1[valid]))),
    "NDRE+ (plaster/lime)":      (NDRE > 0.1)  & valid,
    "Shadow/low radiance":       (N1   < float(np.nanpercentile(N1[valid], 15))) & valid,
    "High-NIR bright soil":      (N1   > float(np.nanpercentile(N1[valid], 75))) & (NDVI < 0.05) & valid,
}

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#d4a574", "#7ecfce", "#a78fcf", "#444466", "#e88c3a"]

for (label, mask), color in zip(region_masks.items(), colors):
    n = mask.sum()
    if n < 100:
        continue
    idx_flat = np.where(mask.reshape(-1))[0]
    sample = np.random.default_rng(42).choice(idx_flat, size=min(5000, n), replace=False)
    ri, ci = np.unravel_index(sample, (H, W))
    spectra = radiance[:, ri, ci]
    mean_s  = spectra.mean(axis=1)
    std_s   = spectra.std(axis=1)
    ax.plot(WAVELENGTHS, mean_s, label=f"{label} (n={n:,})", color=color, lw=2)
    ax.fill_between(WAVELENGTHS, mean_s - std_s, mean_s + std_s, alpha=0.15, color=color)

ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Radiance (W/m²/sr/µm)")
ax.set_title("Mean Spectral Signatures by Land Cover Class", fontweight="bold")
ax.legend(fontsize=7)
ax.grid(alpha=0.3)
ax.set_xticks(WAVELENGTHS)
ax.set_xticklabels([f"{n}nm\n{b}" for n, b in zip(WAVELENGTHS, WV2_BANDS)], fontsize=7)
plt.tight_layout()
plt.savefig(OUT / "05_spectral_signatures.png", bbox_inches="tight")
plt.show()

In [ ]:
# calculate principal variation components and reshape data
# save principal variation maps to disk
flat = radiance.reshape(8, -1).T
valid_flat = valid.reshape(-1)
X = flat[valid_flat].astype(np.float32)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=8)
pca.fit(X_scaled)
explained = pca.explained_variance_ratio_
pc_scores = pca.transform(X_scaled)

print("PCA explained variance:")
for i, ev in enumerate(explained):
    print(f"  PC{i+1}: {ev*100:.1f}%  (cumulative: {explained[:i+1].sum()*100:.1f}%)")

pc_maps = np.full((8, H * W), np.nan, dtype=np.float32)
pc_maps[:, valid_flat] = pc_scores.T
pc_maps = pc_maps.reshape(8, H, W)

fig = plt.figure(figsize=(20, 12))
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.4, wspace=0.3)
fig.suptitle("WV-2 PCA — El Bagawat", fontsize=12, fontweight="bold")

cmaps = ["RdBu_r", "PuOr_r", "BrBG_r", "RdYlGn_r"]
for i in range(4):
    ax = fig.add_subplot(gs[0, i])
    pc = pc_maps[i]
    v = float(np.nanpercentile(np.abs(pc), 98))
    ax.imshow(pc, cmap=cmaps[i], vmin=-v, vmax=v, interpolation="nearest")
    ax.set_title(f"PC{i+1} ({explained[i]*100:.1f}%)", fontsize=9)
    ax.axis("off")

ax_scree = fig.add_subplot(gs[1, :2])
ax_scree.bar(range(1, 9), explained * 100, color="#5b8dd9", alpha=0.8)
ax_scree.plot(range(1, 9), np.cumsum(explained) * 100, "o-", color="#e06c5a", lw=2)
ax_scree.axhline(95, color="gray", ls="--", alpha=0.5)
ax_scree.set_xlabel("PC"); ax_scree.set_ylabel("Variance (%)")
ax_scree.set_title("Scree Plot"); ax_scree.set_xticks(range(1, 9))
ax_scree.grid(alpha=0.3)

ax_load = fig.add_subplot(gs[1, 2:])
loadings = pca.components_[:2].T
for j, name in enumerate(WV2_BANDS):
    ax_load.annotate(name, xy=(loadings[j, 0], loadings[j, 1]), fontsize=8,
                     ha="center", va="center",
                     bbox=dict(boxstyle="round,pad=0.2", fc="#f0f0f0", alpha=0.7))
    ax_load.arrow(0, 0, loadings[j, 0] * 0.9, loadings[j, 1] * 0.9,
                  head_width=0.02, head_length=0.01, fc="#4477aa", ec="#4477aa", alpha=0.7)
ax_load.set_xlim(-1.1, 1.1); ax_load.set_ylim(-1.1, 1.1)
ax_load.axhline(0, color="gray", alpha=0.3); ax_load.axvline(0, color="gray", alpha=0.3)
ax_load.set_xlabel("PC1 loading"); ax_load.set_ylabel("PC2 loading")
ax_load.set_title("Band Loadings (PC1 vs PC2)"); ax_load.grid(alpha=0.3)

ax_scat = fig.add_subplot(gs[2, :2])
n_samp = min(15000, pc_scores.shape[0])
idx_s = np.random.default_rng(0).choice(pc_scores.shape[0], n_samp, replace=False)
ax_scat.scatter(pc_scores[idx_s, 0], pc_scores[idx_s, 1],
                c=pc_scores[idx_s, 2], cmap="RdYlBu", s=0.5, alpha=0.5)
ax_scat.set_xlabel("PC1"); ax_scat.set_ylabel("PC2")
ax_scat.set_title("PC1 vs PC2 (colored by PC3)"); ax_scat.grid(alpha=0.3)

ax_prgb = fig.add_subplot(gs[2, 2:])
pca_rgb = make_rgb(pc_maps[0], pc_maps[1], pc_maps[2], nodata_mask)
ax_prgb.imshow(pca_rgb)
ax_prgb.set_title("PCA False Color (PC1-PC2-PC3)"); ax_prgb.axis("off")

plt.savefig(OUT / "06_pca_analysis.png", bbox_inches="tight")
plt.show()

In [ ]:
# estimate noise levels across the image to separate signal from static
# transform image to group the cleanest signals together
def estimate_noise_cov(cube, valid_mask):
    C, H_c, W_c = cube.shape
    d = cube[:, :, 1:] - cube[:, :, :-1]
    valid2d = valid_mask[:, 1:] & valid_mask[:, :-1]
    d_flat = d.reshape(C, -1).T[valid2d.reshape(-1)]
    return np.cov(d_flat.T) / 2.0

noise_cov = estimate_noise_cov(radiance, valid.reshape(H, W))
evals, evecs = np.linalg.eigh(noise_cov)
evals = np.maximum(evals, 1e-10)
W_mat = (evecs / np.sqrt(evals)).T

X_white = X_scaled @ W_mat.T

mnf = PCA(n_components=8)
mnf.fit(X_white)
mnf_scores = mnf.transform(X_white)
snr_approx = mnf.explained_variance_

print("MNF components — higher variance = more signal content:")
for i, s in enumerate(snr_approx):
    print(f"  MNF{i+1}: approx signal var = {s:.3f}")

mnf_maps = np.full((8, H * W), np.nan, dtype=np.float32)
mnf_maps[:, valid_flat] = mnf_scores.T
mnf_maps = mnf_maps.reshape(8, H, W)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("MNF Transform — El Bagawat (MNF1 = most signal, MNF8 = noise)", fontsize=11, fontweight="bold")
for i, ax in enumerate(axes.flat):
    m = mnf_maps[i]
    v = float(np.nanpercentile(np.abs(m[~nodata_mask]), 98))
    ax.imshow(m, cmap="RdBu_r", vmin=-v, vmax=v, interpolation="nearest")
    ax.set_title(f"MNF{i+1}  var={snr_approx[i]:.2f}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "07_mnf_components.png", bbox_inches="tight")
plt.show()

In [ ]:
# keep most informative variation layers and discard random noise
# save noise filtered feature maps to file
n_pcs_keep = int(np.searchsorted(np.cumsum(explained), 0.95)) + 1
print(f"Using first {n_pcs_keep} PCs ({np.cumsum(explained)[n_pcs_keep-1]*100:.1f}% variance)")

X_pca = pc_scores[:, :n_pcs_keep]

N_CLUSTERS = 8
km = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=5, batch_size=200_000, max_iter=500)
labels = km.fit_predict(X_pca)

label_map = np.full(H * W, -1, dtype=np.int16)
label_map[valid_flat] = labels
label_map = label_map.reshape(H, W)

cluster_spectra = {}
for k in range(N_CLUSTERS):
    mask_k = valid_flat & (label_map.reshape(-1) == k)
    cluster_spectra[k] = radiance.reshape(8, -1)[:, mask_k].mean(axis=1)

sorted_cls = sorted(range(N_CLUSTERS), key=lambda k: cluster_spectra[k][band_idx["NIR1"]])
cmap_cls = plt.colormaps["tab10"].resampled(N_CLUSTERS)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(f"K-Means Spectral Clustering ({N_CLUSTERS} classes in PCA space)", fontsize=11, fontweight="bold")

lm_disp = label_map.astype(float)
lm_disp[label_map == -1] = np.nan
axes[0].imshow(lm_disp, cmap=cmap_cls, vmin=0, vmax=N_CLUSTERS - 1, interpolation="nearest")
axes[0].set_title("Cluster Map"); axes[0].axis("off")
patches = [Patch(color=cmap_cls(k), label=f"Class {k}") for k in range(N_CLUSTERS)]
axes[0].legend(handles=patches, loc="lower right", fontsize=7, framealpha=0.8)

for k in sorted_cls:
    spec = cluster_spectra[k]
    n_px = (label_map == k).sum()
    axes[1].plot(WAVELENGTHS, spec, label=f"Class {k} (n={n_px:,})", color=cmap_cls(k), lw=2)

axes[1].set_xlabel("Wavelength (nm)"); axes[1].set_ylabel("Radiance (W/m²/sr/µm)")
axes[1].set_title("Mean Spectrum per Cluster"); axes[1].legend(fontsize=7)
axes[1].set_xticks(WAVELENGTHS)
axes[1].set_xticklabels([f"{n}nm\n{b}" for n, b in zip(WAVELENGTHS, WV2_BANDS)], fontsize=7)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "08_kmeans_clustering.png", bbox_inches="tight")
plt.show()

In [ ]:
# pick a random sample of points across the study area
# group similar points together using two different sorting tools
# plot charts showing how the different ground points cluster
N_TSNE = 20000
sub_idx = np.random.default_rng(7).choice(X_pca.shape[0], min(N_TSNE, X_pca.shape[0]), replace=False)
X_sub   = X_pca[sub_idx]
lab_sub = labels[sub_idx]

print(f"Running t-SNE on {len(sub_idx)} pixels ...")
tsne = TSNE(n_components=2, perplexity=50, learning_rate=200, n_iter=1000, random_state=42, n_jobs=-1)
emb = tsne.fit_transform(X_sub)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("t-SNE Spectral Embedding — El Bagawat", fontsize=11, fontweight="bold")

axes[0].scatter(emb[:, 0], emb[:, 1], c=lab_sub, cmap=cmap_cls, s=1, alpha=0.6, vmin=0, vmax=N_CLUSTERS - 1)
axes[0].set_title("Colored by K-Means cluster")
axes[0].set_xlabel("t-SNE 1"); axes[0].set_ylabel("t-SNE 2"); axes[0].axis("equal")

ndvi_sub = NDVI.reshape(-1)[valid_flat][sub_idx]
sc2 = axes[1].scatter(emb[:, 0], emb[:, 1], c=ndvi_sub, cmap="RdYlGn", s=1, alpha=0.6, vmin=-0.3, vmax=0.3)
plt.colorbar(sc2, ax=axes[1], fraction=0.046, label="NDVI")
axes[1].set_title("Colored by NDVI")
axes[1].set_xlabel("t-SNE 1"); axes[1].axis("equal")

bsi_sub = BSI.reshape(-1)[valid_flat][sub_idx]
sc3 = axes[2].scatter(emb[:, 0], emb[:, 1], c=bsi_sub, cmap="YlOrBr", s=1, alpha=0.6, vmin=-0.3, vmax=0.5)
plt.colorbar(sc3, ax=axes[2], fraction=0.046, label="BSI")
axes[2].set_title("Colored by BSI")
axes[2].set_xlabel("t-SNE 1"); axes[2].axis("equal")

plt.tight_layout()
plt.savefig(OUT / "09_tsne_spectral_space.png", bbox_inches="tight")
plt.show()

In [ ]:
# load newer high detail satellite image if available
# crop high detail image to match our study area
# compare details between older and newer satellite pictures
lg_path = DATA / "LandInfo_2026/LG04 HD15/LG4_26JAN01124636-S3DS-050535636010_01_P001.tif"
LG_AVAILABLE = False
try:
    print(f"LandInfo file: {lg_path.stat().st_size / 1e9:.2f} GB")
    with rasterio.open(lg_path) as src:
        lg_crs = src.crs
        lg_tf  = src.transform
        lg_H, lg_W = src.height, src.width
        print(f"LandInfo: {lg_W}x{lg_H} px, {src.res[0]:.4f}m res, CRS: {lg_crs}")
        cx, cy = lg_W // 2, lg_H // 2
        win_size = 2000
        window = rasterio.windows.Window(cx - win_size//2, cy - win_size//2, win_size, win_size)
        lg_crop = src.read(window=window)

    lg_rgb = np.stack([lg_crop[0], lg_crop[1], lg_crop[2]], axis=2)
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(lg_rgb, interpolation="nearest")
    ax.set_title("LandInfo 2026 — Maxar Legion 15cm RGB (2000x2000 px center crop)", fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT / "10_landinfo_rgb_crop.png", bbox_inches="tight")
    plt.show()
    LG_AVAILABLE = True

except FileNotFoundError:
    print(f"WARNING: LandInfo file not found at:\n  {lg_path}\n  Skipping high-res comparison cells.")

In [ ]:
# automatically group ground pixels into five surface types
# sort surface types from darkest shadow to brightest ground
# save maps showing the different ground surface types
k_dark   = sorted_cls[0]
k_bright = sorted_cls[-1]
ref_dark   = cluster_spectra[k_dark].astype(np.float32)
ref_bright = cluster_spectra[k_bright].astype(np.float32)

X_full = radiance.reshape(8, -1).T.astype(np.float32)
sam_dark   = spectral_angle(X_full, ref_dark).reshape(H, W)
sam_bright = spectral_angle(X_full, ref_bright).reshape(H, W)
sam_dark[nodata_mask]   = np.nan
sam_bright[nodata_mask] = np.nan

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Spectral Angle Mapper — El Bagawat", fontsize=11, fontweight="bold")
axes[0].imshow(tc); axes[0].set_title("True Color reference"); axes[0].axis("off")

im1 = axes[1].imshow(sam_dark, cmap="hot_r", vmin=0, vmax=30, interpolation="nearest")
axes[1].set_title(f"SAM to Class {k_dark} (dark/structure)\n<10° = strong match", fontsize=8)
axes[1].axis("off"); plt.colorbar(im1, ax=axes[1], fraction=0.046, label="Angle (°)")

im2 = axes[2].imshow(sam_bright, cmap="cool_r", vmin=0, vmax=30, interpolation="nearest")
axes[2].set_title(f"SAM to Class {k_bright} (bright sand)\n<10° = strong match", fontsize=8)
axes[2].axis("off"); plt.colorbar(im2, ax=axes[2], fraction=0.046, label="Angle (°)")

plt.tight_layout()
plt.savefig(OUT / "11_spectral_angle_mapper.png", bbox_inches="tight")
plt.show()

In [ ]:
# compare cluster results against high detail map images
# check how many pixels of each ground type appear across the site
if LG_AVAILABLE:
    with rasterio.open(lg_path) as src:
        lg_bounds_wv2 = transform_bounds(src.crs, crs, src.bounds.left, src.bounds.bottom, src.bounds.right, src.bounds.top)

    left   = max(mosaic_tf.c, lg_bounds_wv2[0])
    bottom = max(mosaic_tf.f + H * mosaic_tf.e, lg_bounds_wv2[1])
    right  = min(mosaic_tf.c + W * mosaic_tf.a, lg_bounds_wv2[2])
    top    = min(mosaic_tf.f, lg_bounds_wv2[3])

    print(f"Overlap (WV-2 CRS): L={left:.0f} B={bottom:.0f} R={right:.0f} T={top:.0f}")

    if right <= left or top <= bottom:
        print("No geographic overlap between WV-2 and LandInfo.")
    else:
        dst_res = 2.0
        dst_W = int((right - left) / dst_res)
        dst_H = int((top - bottom) / dst_res)
        dst_tf = from_bounds(left, bottom, right, top, dst_W, dst_H)

        wv2_rgb_ov = np.zeros((3, dst_H, dst_W), dtype=np.float32)
        for out_i, bi in enumerate([band_idx["Red"], band_idx["Green"], band_idx["Blue"]]):
            reproject(source=radiance[bi], destination=wv2_rgb_ov[out_i],
                      src_transform=mosaic_tf, src_crs=crs,
                      dst_transform=dst_tf, dst_crs=crs,
                      resampling=Resampling.bilinear)

        lg_ov = np.zeros((3, dst_H, dst_W), dtype=np.float32)
        with rasterio.open(lg_path) as src:
            for bi in range(3):
                reproject(source=src.read(bi + 1).astype(np.float32), destination=lg_ov[bi],
                          src_transform=src.transform, src_crs=src.crs,
                          dst_transform=dst_tf, dst_crs=crs,
                          resampling=Resampling.bilinear)

        wv2_n = np.stack([norm01(wv2_rgb_ov[i]) for i in range(3)], axis=0)
        lg_n  = np.stack([norm01(lg_ov[i])      for i in range(3)], axis=0)

        change_mag = np.linalg.norm(lg_n - wv2_n, axis=0).astype(np.float32)
        valid_cd   = (wv2_rgb_ov[0] > 0) & (lg_ov[0] > 0)
        change_mag[~valid_cd] = np.nan

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle("Change Detection: WV-2 2018 vs LandInfo 2026", fontsize=11, fontweight="bold")

        wv2_d = np.stack([norm01(wv2_rgb_ov[i]) for i in range(3)], axis=2); wv2_d[~valid_cd] = 0
        lg_d  = np.stack([norm01(lg_ov[i])      for i in range(3)], axis=2); lg_d[~valid_cd]  = 0

        axes[0].imshow(wv2_d); axes[0].set_title("WV-2 2018 (RGB 2m)"); axes[0].axis("off")
        axes[1].imshow(lg_d);  axes[1].set_title("LandInfo 2026 (RGB 2m)"); axes[1].axis("off")
        im = axes[2].imshow(change_mag, cmap="RdYlGn_r", vmin=0, vmax=float(np.nanpercentile(change_mag, 95)), interpolation="nearest")
        axes[2].set_title("Change Magnitude 2018 -> 2026\nRed = large spectral change"); axes[2].axis("off")
        plt.colorbar(im, ax=axes[2], fraction=0.046, label="|delta| normalized")

        plt.tight_layout()
        plt.savefig(OUT / "12_change_detection.png", bbox_inches="tight")
        plt.show()

In [ ]:
# calculate average light reflection across bands for each ground type
# draw line chart comparing how different surfaces reflect light
# measure how well separated the different ground clusters are
corr = np.corrcoef(X.T)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("WV-2 Band Correlation Analysis", fontsize=11, fontweight="bold")

im = axes[0].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_xticks(range(8)); axes[0].set_yticks(range(8))
axes[0].set_xticklabels(WV2_BANDS, rotation=45, ha="right", fontsize=8)
axes[0].set_yticklabels(WV2_BANDS, fontsize=8)
axes[0].set_title("Band Correlation Matrix")
for i in range(8):
    for j in range(8):
        axes[0].text(j, i, f"{corr[i,j]:.2f}", ha="center", va="center", fontsize=6, color="black" if abs(corr[i,j]) < 0.7 else "white")
plt.colorbar(im, ax=axes[0], fraction=0.046)

sub2 = np.random.default_rng(3).choice(X.shape[0], min(8000, X.shape[0]), replace=False)
axes[1].scatter(X[sub2, band_idx["Red"]], X[sub2, band_idx["NIR1"]], c=X[sub2, band_idx["RedEdge"]], cmap="RdYlGn", s=0.8, alpha=0.5)
axes[1].set_xlabel("Red radiance"); axes[1].set_ylabel("NIR1 radiance")
axes[1].set_title("Red vs NIR1 (colored by RedEdge)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / "13_band_correlation.png", bbox_inches="tight")
plt.show()

In [ ]:
# draw visual grid comparing all seven calculated indices
# save index summary picture to output folder
print("Saving GeoTIFFs ...")
save_tif(label_map.astype(np.int16), OUT / "cluster_map_8cls.tif", mosaic_tf, crs, nodata=-1, dtype=np.int16)

for i in range(3):
    m = pc_maps[i].copy()
    m[np.isnan(m)] = -9999.0
    save_tif(m, OUT / f"PC{i+1}.tif", mosaic_tf, crs, nodata=-9999.0)

for arr, name in [(NDRE, "NDRE"), (BSI, "BSI"), (NDVI, "NDVI")]:
    a = np.where(np.isnan(arr), -9999.0, arr).astype(np.float32)
    save_tif(a, OUT / f"{name}.tif", mosaic_tf, crs, nodata=-9999.0)

sam_d = np.where(np.isnan(sam_dark), -9999.0, sam_dark).astype(np.float32)
save_tif(sam_d, OUT / "SAM_dark_class.tif", mosaic_tf, crs, nodata=-9999.0)

print("  Saving MNF bands (noise-whitened PCA) ...")
for i in range(4):
    m = mnf_maps[i].copy()
    m[np.isnan(m)] = -9999.0
    save_tif(m, OUT / f"MNF{i+1}.tif", mosaic_tf, crs, nodata=-9999.0)

fdi_save = np.where(np.isnan(FDI), -9999.0, FDI).astype(np.float32)
save_tif(fdi_save, OUT / "FDI.tif", mosaic_tf, crs, nodata=-9999.0)
print(f"All outputs saved to: {OUT}")

In [ ]:
# save all seven calculated index maps as individual image files
with rasterio.open(OUT_SPEC / "cluster_map_8cls.tif") as src:
    clust_raw = src.read(1).astype(np.int16)
    clust_tf  = src.transform
    clust_crs = src.crs

bsi, _, bsi_tf, bsi_crs = load_tif(OUT_SPEC / "BSI.tif")
H, W = bsi.shape
valid_mask = ~np.isnan(bsi)

N_CLUSTERS = 8
cluster_bsi = {}
for k in range(N_CLUSTERS):
    mask = (clust_raw == k) & valid_mask
    vals = bsi[mask]
    cluster_bsi[k] = float(np.nanmean(vals)) if vals.size > 0 else np.nan

print("Mean BSI per cluster (lower = more compacted):")
for k in range(N_CLUSTERS):
    n = int((clust_raw == k).sum())
    print(f"  Class {k}: BSI={cluster_bsi[k]:.4f}  n={n:,}")

SEMANTIC = {3: "shadow_alley", 6: "shaded_compacted", 4: "bright_anomaly"}
DEFAULT_SEMANTIC = "desert_sand"

for k in range(N_CLUSTERS):
    n = int((clust_raw == k).sum())
    label = SEMANTIC.get(k, DEFAULT_SEMANTIC)
    print(f"  Class {k} -> {label:20s}  (n={n:,})")

cluster_path_mask = np.isin(clust_raw, [3, 6]) & valid_mask
print(f"Path-class pixels: {cluster_path_mask.sum():,}")

In [ ]:
# identify shadow and tomb structures using dark cluster areas
# identify walking paths using soil index and bright surface maps
# identify surviving plant life using greenness index
# save feature detection maps to disk
fdi, _, fdi_tf, fdi_crs = load_tif(OUT_SPEC / "FDI.tif")

bsi_med = float(np.nanmedian(bsi[valid_mask]))
fdi_med = float(np.nanmedian(fdi[valid_mask]))

bsi_score = sigmoid(-(bsi - bsi_med) * 25.0)
fdi_score = sigmoid(-(fdi - fdi_med) * 0.08)
bsi_score[~valid_mask] = np.nan
fdi_score[~valid_mask] = np.nan

material_score = 0.60 * np.nan_to_num(bsi_score) + 0.40 * np.nan_to_num(fdi_score)
material_score[~valid_mask] = np.nan

save_tif(material_score, OUT_SPEC / "path_material_score.tif", bsi_tf, bsi_crs)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Attempt A - Material Score (BSI + FDI)", fontsize=11, fontweight="bold")
for ax, arr, title, cmap in zip(axes, [bsi, fdi, material_score], ["BSI (low=compacted)", "FDI (low=path-like)", "Material Score"], ["YlOrBr_r", "Spectral_r", "hot"]):
    p2, p98 = np.nanpercentile(arr, [2, 98])
    im = ax.imshow(arr, cmap=cmap, vmin=p2, vmax=p98, interpolation="nearest")
    ax.set_title(title); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT_SPEC / "16A_material_score.png", bbox_inches="tight")
plt.show()

In [ ]:
# load soil index map and principal variation map
# check how strongly soil index matches the main variation pattern
# plot chart showing agreement between different measurements
mnf1, _, mnf_tf, mnf_crs = load_tif(OUT_SPEC / "MNF1.tif")
mnf2, _, _, _ = load_tif(OUT_SPEC / "MNF2.tif")
mnf3, _, _, _ = load_tif(OUT_SPEC / "MNF3.tif")

mnf1_n = norm01(np.nan_to_num(mnf1, nan=0.0))
mnf2_n = norm01(np.nan_to_num(mnf2, nan=0.0))
mnf3_n = norm01(np.nan_to_num(mnf3, nan=0.0))

FREQUENCIES  = [0.15,  0.25,  0.40]
FREQ_WEIGHTS = [0.25,  0.45,  0.30]
MNF_INPUTS   = [(mnf1_n, 0.55), (mnf2_n, 0.30), (mnf3_n, 0.15)]

gabor_acc = np.zeros((H, W), dtype=np.float32)
n_total   = len(FREQUENCIES) * len(MNF_INPUTS)
done = 0
for freq, fw in zip(FREQUENCIES, FREQ_WEIGHTS):
    print(f"  Gabor frequency={freq:.2f} ...")
    for mnf_img, mw in MNF_INPUTS:
        resp   = max_gabor_responses(mnf_img, freq, n_angles=9)
        resp_n = norm01(np.nan_to_num(resp, nan=0.0))
        gabor_acc += fw * mw * resp_n
        done += 1
        print(f"    [{done}/{n_total}]")

gabor_response = norm01(gabor_acc)
gabor_response[~valid_mask] = np.nan
save_tif(gabor_response, OUT_SPEC / "path_gabor_response.tif", mnf_tf, mnf_crs)

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle("Attempt B - Multi-scale MNF Gabor Bank", fontsize=11, fontweight="bold")
for ax, arr, title, cmap in zip(axes, [mnf1_n, mnf2_n, mnf3_n, gabor_response], ["MNF1 (var=0.60)", "MNF2 (var=0.21)", "MNF3 (var=0.03)", "Gabor ensemble"], ["RdBu_r", "RdBu_r", "RdBu_r", "hot"]):
    p2, p98 = np.nanpercentile(arr, [2, 98])
    im = ax.imshow(arr, cmap=cmap, vmin=p2, vmax=p98, interpolation="nearest")
    ax.set_title(title, fontsize=8); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT_SPEC / "16B_gabor_response.png", bbox_inches="tight")
plt.show()

In [ ]:
# check where multiple different index formulas agree on path locations
# add up signals across indices to find strongest candidate walking routes
# draw and save map showing where surface signals agree
coherence_acc = np.zeros((H, W), dtype=np.float32)
for sigma_val in [2.0, 3.0]:
    A_xx, A_xy, A_yy = structure_tensor(mnf1_n.astype(np.float64), sigma=sigma_val, mode="reflect")
    eig1, eig2 = structure_tensor_eigenvalues((A_xx, A_xy, A_yy))
    eig1 = np.maximum(eig1, 0.0)
    eig2 = np.maximum(eig2, 0.0)
    coh  = ((eig1 - eig2) / (eig1 + eig2 + 1e-8)) ** 2
    coherence_acc = np.maximum(coherence_acc, norm01(coh).astype(np.float32))

coherence_acc[~valid_mask] = np.nan
save_tif(coherence_acc, OUT_SPEC / "path_coherence.tif", mnf_tf, mnf_crs)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Attempt C - Structure Tensor Coherence on MNF1", fontsize=11, fontweight="bold")
axes[0].imshow(mnf1_n, cmap="gray", interpolation="nearest"); axes[0].set_title("MNF1"); axes[0].axis("off")
im = axes[1].imshow(coherence_acc, cmap="hot", interpolation="nearest", vmin=0, vmax=np.nanpercentile(coherence_acc, 98))
axes[1].set_title("Coherence (max at sigma=2,3)"); axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.savefig(OUT_SPEC / "16C_coherence.png", bbox_inches="tight")
plt.show()

In [ ]:
# compare old and new satellite images to see which ground areas changed
# highlight areas where surface brightness remained stable over time
# draw and save ground stability map
stab_path = OUT_SPEC / "surface_stability.tif"
if stab_path.exists():
    stability, _, stab_tf, stab_crs = load_tif(stab_path)
    print("Loaded existing surface_stability.tif")
else:
    print("Building BSI-proxy stability (cluster path mask x low-BSI weighting) ...")
    stab_proxy = cluster_path_mask.astype(np.float32) * np.nan_to_num(1.0 - bsi_score)
    stability  = norm01(gaussian_filter(stab_proxy, sigma=1.5))
    stability[~valid_mask] = np.nan
    stab_tf, stab_crs = bsi_tf, bsi_crs
    save_tif(stability, stab_path, stab_tf, stab_crs)

fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(stability, cmap="YlGn", interpolation="nearest", vmin=0, vmax=np.nanpercentile(stability, 98))
ax.set_title("Surface Stability (0=sand-drifted, 1=stable)"); ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT_SPEC / "17_stability.png", bbox_inches="tight")
plt.show()

In [ ]:
# combine surface path detection with ground stability measurements
# enhance continuous walking trails while removing scattered noise
# save combined walking path strength map
path_signal = (
    0.40 * np.nan_to_num(gabor_response)
    + 0.30 * np.nan_to_num(material_score)
    + 0.15 * np.nan_to_num(coherence_acc)
    + 0.15 * np.nan_to_num(stability)
)

path_signal_s = gaussian_filter(path_signal, sigma=1.0)
path_signal_s[~valid_mask] = np.nan
path_signal_f = norm01(path_signal_s)
path_signal_f[~valid_mask] = np.nan
save_tif(path_signal_f, OUT_SPEC / "path_signal.tif", bsi_tf, bsi_crs)

SITE_BUFFER_M = 15.0
_fp_path = OUT_VEC / "footprints_attributed.geojson"
if _fp_path.exists():
    _fp = gpd.read_file(str(_fp_path)).to_crs(bsi_crs)
    _site_hull = _fp.unary_union.convex_hull.buffer(SITE_BUFFER_M)
else:
    _net_fb = gpd.read_file(str(OUT_VEC / "path_network_obstacle_free.geojson")).to_crs(bsi_crs)
    _site_hull = _net_fb.unary_union.convex_hull.buffer(SITE_BUFFER_M)

_hull_bx = _site_hull.bounds
SITE_MINX, SITE_MINY, SITE_MAXX, SITE_MAXY = _hull_bx
print(f"Site hull clip (UTM, +{SITE_BUFFER_M}m buffer): E {SITE_MINX:.0f} – {SITE_MAXX:.0f} | N {SITE_MINY:.0f} – {SITE_MAXY:.0f}")

def _hull_pixel_mask(tf, shape, hull_geom):
    return ~geometry_mask([mapping(hull_geom)], transform=tf, invert=False, out_shape=shape)

site_pixel_mask = _hull_pixel_mask(bsi_tf, path_signal_f.shape, _site_hull)
print(f"Site mask covers {site_pixel_mask.sum():,} pixels ({site_pixel_mask.sum()*4/1e4:.1f} ha)")

THRESHOLD = 0.45
path_bin   = (path_signal_f >= THRESHOLD) & valid_mask & site_pixel_mask
path_clean = remove_small_objects(path_bin, min_size=50)
save_tif(path_clean.astype(np.float32), OUT_SPEC / "path_candidate_mask.tif", bsi_tf, bsi_crs, nodata=-1)

skel = skeletonize(path_clean)
print(f"Skeleton pixels (site-clipped): {skel.sum():,} (~{skel.sum()*2:.0f} m)")

skel_gdf = skeleton_to_geojson(skel, bsi_tf, bsi_crs)
if not skel_gdf.empty:
    skel_gdf = skel_gdf[skel_gdf["n_pixels"] >= 10].reset_index(drop=True)
    skel_gdf.to_file(OUT_VEC / "path_skeleton_lines.geojson", driver="GeoJSON")
    print(f"Skeleton segments (>=10 px): {len(skel_gdf)}")
else:
    print("WARNING: skeleton vectorisation returned empty GeoDataFrame")

_net18_path = OUT_VEC / "path_network_obstacle_free.geojson"
_net18 = gpd.read_file(str(_net18_path)).to_crs(bsi_crs) if _net18_path.exists() else None

def _utm_to_px(tf, x, y):
    c, r = ~tf * (x, y)
    return int(r), int(c)

_r0, _c0 = _utm_to_px(bsi_tf, SITE_MINX, SITE_MAXY)
_r1, _c1 = _utm_to_px(bsi_tf, SITE_MAXX, SITE_MINY)
_r0 = max(0, _r0); _c0 = max(0, _c0)
_r1 = min(path_signal_f.shape[0]-1, _r1)
_c1 = min(path_signal_f.shape[1]-1, _c1)

fig, axes = plt.subplots(1, 4, figsize=(22, 7))
fig.suptitle("CELL 18 - Ensemble Path Signal (site-clipped)", fontsize=11, fontweight="bold")

_crop = lambda a: a[_r0:_r1+1, _c0:_c1+1]
_site_extent = [SITE_MINX, SITE_MAXX, SITE_MINY, SITE_MAXY]

panels = [
    (_crop(path_signal_f),            "Path Signal [0,1]",       "hot"),
    (_crop(path_clean.astype(float)), "Binary mask (site-clip)", "gray"),
    (_crop(skel.astype(float)),       "Skeleton",                "gray"),
    (_crop(path_signal_f),            "Signal + skeleton + network", "hot"),
]
for ax, (arr, title, cmap) in zip(axes, panels):
    arr_valid = arr[~np.isnan(arr)]
    p2, p98 = np.nanpercentile(arr_valid, [2, 98]) if arr_valid.size > 0 else (0, 1)
    ax.imshow(arr, cmap=cmap, vmin=p2, vmax=p98, interpolation="nearest", extent=_site_extent, origin="upper")
    ax.set_title(title, fontsize=8)
    ax.set_xlabel("Easting (m)", fontsize=6); ax.set_ylabel("Northing (m)", fontsize=6)
    ax.tick_params(labelsize=6); ax.set_aspect("equal")

if skel.any():
    sk_r, sk_c = np.where(skel)
    sk_xs, sk_ys = rasterio.transform.xy(bsi_tf, sk_r.tolist(), sk_c.tolist())
    axes[3].scatter(sk_xs, sk_ys, s=0.4, c="cyan", alpha=0.8, zorder=4)
if _net18 is not None:
    _net18.plot(ax=axes[3], color="white", linewidth=1.2, alpha=0.85, linestyle="--", zorder=5, label="Door-road network")
    axes[3].legend(fontsize=6, loc="lower right")

plt.tight_layout()
plt.savefig(OUT_SPEC / "18_path_signal_skeleton.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# load existing road and corridor map layers
# calculate how strongly each road segment matches satellite detections
# calculate overall connection importance and density for every path
# save scored path network to file
net_path = OUT_VEC / "path_network_obstacle_free.geojson"
if not net_path.exists():
    print(f"WARNING: {net_path} not found - skipping Cell 19")
else:
    network_raw = gpd.read_file(str(net_path))
    network = network_raw.to_crs(bsi_crs)
    print(f"Loaded network: {len(network)} edges (reprojected to {bsi_crs})")

    ps_arr, _, ps_tf, _ = load_tif(OUT_SPEC / "path_signal.tif")

    circ_path = _base / "outputs" / "circuit_current_pairwise.tif"
    if circ_path.exists():
        with rasterio.open(circ_path) as src:
            circ_arr  = src.read(1).astype(np.float32)
            circ_tf   = src.transform
            circ_crs2 = src.crs
        circ_H, circ_W = circ_arr.shape
        circ_has_data = True
        print(f"Circuit raster CRS: {circ_crs2}")
    else:
        circ_has_data = False
        print("circuit_current_pairwise.tif not found - circuit score = 0")

    records = []
    for idx, row in network.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue

        sp_vals      = sample_raster_along_line(geom, ps_arr, ps_tf, 2.0)
        stab_vals    = sample_raster_along_line(geom, stability, stab_tf, 2.0)
        spec_support = float(np.nanmean(sp_vals))   if not np.all(np.isnan(sp_vals)) else 0.0
        stab_mean    = float(np.nanmean(stab_vals)) if not np.all(np.isnan(stab_vals)) else 0.0

        cl_vals = sample_raster_along_line(geom, clust_raw.astype(np.float32), clust_tf, 2.0)
        cl_ok   = cl_vals[~np.isnan(cl_vals) & (cl_vals >= 0)]
        if len(cl_ok):
            unique_cl, counts_cl = np.unique(cl_ok, return_counts=True)
            spec_class = SEMANTIC.get(int(unique_cl[np.argmax(counts_cl)]), DEFAULT_SEMANTIC)
        else:
            spec_class = "nodata"

        if circ_has_data:
            cent = geom.centroid
            if circ_crs2 != bsi_crs:
                cent_gs = gpd.GeoSeries([cent], crs=bsi_crs).to_crs(circ_crs2)
                cx, cy = cent_gs.iloc[0].x, cent_gs.iloc[0].y
            else:
                cx, cy = cent.x, cent.y
            cc_f, cr_f = ~circ_tf * (cx, cy)
            cr2, cc2 = int(cr_f), int(cc_f)
            circ_v = float(circ_arr[cr2, cc2]) if 0 <= cr2 < circ_H and 0 <= cc2 < circ_W else 0.0
        else:
            circ_v = 0.0

        edge_bt = float(row["betweenness"]) if "betweenness" in network.columns and not pd.isna(row["betweenness"]) else 0.0

        r = {k: row[k] for k in network.columns if k != "geometry"}
        r.update({
            "spec_support":     round(spec_support, 4),
            "spec_class":       spec_class,
            "stability_mean":   round(stab_mean, 4),
            "circuit_density":  round(circ_v, 4),
            "betweenness_edge": round(edge_bt, 6),
            "geometry": geom,
        })
        records.append(r)

    scored = gpd.GeoDataFrame(records, crs=bsi_crs)

    for col, dest in [("spec_support", "spec_support_norm"), ("stability_mean", "stability_norm"), ("circuit_density", "circuit_norm"), ("betweenness_edge", "betweenness_norm")]:
        mx = scored[col].max()
        scored[dest] = (scored[col] / mx).clip(0, 1) if mx > 0 else 0.0

    scored["arch_score"] = (
        0.40 * scored["spec_support_norm"]
        + 0.15 * scored["stability_norm"]
        + 0.25 * scored["betweenness_norm"]
        + 0.20 * scored["circuit_norm"]
    ).round(4)

    scored["confidence"] = scored["arch_score"].apply(lambda s: "high" if s >= 0.60 else ("moderate" if s >= 0.40 else "speculative"))
    scored.to_file(OUT_VEC / "path_network_spectral_scored.geojson", driver="GeoJSON")
    print(f"Saved {len(scored)} scored edges")
    print(scored["confidence"].value_counts().to_string())

    skel_file_19 = OUT_VEC / "path_skeleton_lines.geojson"
    if skel_file_19.exists():
        skel_gdf_19 = gpd.read_file(str(skel_file_19))
        if skel_gdf_19.crs != bsi_crs:
            skel_gdf_19 = skel_gdf_19.to_crs(bsi_crs)
        skel_sp, skel_st = [], []
        for _, srow in skel_gdf_19.iterrows():
            sv = sample_raster_along_line(srow.geometry, ps_arr, ps_tf, 2.0)
            tv = sample_raster_along_line(srow.geometry, stability, stab_tf, 2.0)
            skel_sp.append(float(np.nanmean(sv))   if not np.all(np.isnan(sv)) else 0.0)
            skel_st.append(float(np.nanmean(tv))   if not np.all(np.isnan(tv)) else 0.0)
        skel_gdf_19["spec_support"]   = skel_sp
        skel_gdf_19["stability_mean"] = skel_st
        skel_gdf_19["skel_score"]     = (0.70 * np.array(skel_sp) + 0.30 * np.array(skel_st)).round(4)
        print(f"Scored {len(skel_gdf_19)} skeleton segments as secondary layer")
        has_skel_19 = True
    else:
        has_skel_19 = False

    with rasterio.open(OUT_SPEC / "path_signal.tif") as _src:
        _b = _src.bounds
    site_extent = [SITE_MINX, SITE_MAXX, SITE_MINY, SITE_MAXY]
    full_extent = [_b.left, _b.right, _b.bottom, _b.top]

    mnf1_bg_path = OUT_SPEC / "MNF1.tif"
    if mnf1_bg_path.exists():
        with rasterio.open(mnf1_bg_path) as _src:
            _mnf1_bg = _src.read(1).astype(np.float32)
        _p2, _p98 = np.nanpercentile(_mnf1_bg[~np.isnan(_mnf1_bg)], [2, 98])
        bg_arr = np.clip((_mnf1_bg - _p2) / (_p98 - _p2 + 1e-9), 0, 1)
    else:
        bg_arr = path_signal_f

    n_panels = 3 if has_skel_19 else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(8 * n_panels, 11))
    if n_panels == 2:
        axes = list(axes)
    fig.suptitle("CELL 19 - Spectral-Scored Corridor Network", fontsize=12, fontweight="bold")

    panel_idx = 0
    if has_skel_19:
        ax = axes[panel_idx]; panel_idx += 1
        ax.imshow(bg_arr, cmap="gray", origin="upper", extent=full_extent, vmin=0, vmax=1, alpha=0.60)
        skel_gdf_19.plot(ax=ax, column="skel_score", cmap="plasma", vmin=0, vmax=1, linewidth=2.0, legend=True, legend_kwds={"label": "Skel score", "shrink": 0.6})
        ax.set_xlim(SITE_MINX, SITE_MAXX); ax.set_ylim(SITE_MINY, SITE_MAXY)
        ax.set_title("Skeleton — Spectral+Stability Score", fontsize=9); ax.set_aspect("equal")
        ax.set_xlabel("Easting (m)", fontsize=7); ax.set_ylabel("Northing (m)", fontsize=7)

    ax = axes[panel_idx]; panel_idx += 1
    ax.imshow(bg_arr, cmap="gray", origin="upper", extent=full_extent, vmin=0, vmax=1, alpha=0.60)
    scored.plot(ax=ax, column="arch_score", cmap="RdYlGn", vmin=0, vmax=1, linewidth=2.5, legend=True, legend_kwds={"label": "Arch score", "shrink": 0.6})
    ax.set_xlim(SITE_MINX, SITE_MAXX); ax.set_ylim(SITE_MINY, SITE_MAXY)
    ax.set_title("Door-Road Network — Archaeological Score", fontsize=9); ax.set_aspect("equal")
    ax.set_xlabel("Easting (m)", fontsize=7)

    ax = axes[panel_idx]; panel_idx += 1
    ax.imshow(bg_arr, cmap="gray", origin="upper", extent=full_extent, vmin=0, vmax=1, alpha=0.60)
    scored.plot(ax=ax, column="spec_support", cmap="RdYlGn", vmin=0, vmax=1, linewidth=2.5, legend=True, legend_kwds={"label": "Spectral support", "shrink": 0.6})
    ax.set_xlim(SITE_MINX, SITE_MAXX); ax.set_ylim(SITE_MINY, SITE_MAXY)
    ax.set_title("Door-Road Network — Spectral Support", fontsize=9); ax.set_aspect("equal")
    ax.set_xlabel("Easting (m)", fontsize=7)

    plt.tight_layout()
    plt.savefig(OUT_SPEC / "19_scored_network.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# search for continuous walking paths that are missing from older maps
# check alignment and length to verify new candidate routes
# draw and save map comparing known roads against new discoveries
# save newly identified walking corridors to file
skel_file = OUT_VEC / "path_skeleton_lines.geojson"
net_file  = OUT_VEC / "path_network_obstacle_free.geojson"

if not skel_file.exists() or not net_file.exists():
    print("Missing input files - skipping Cell 20")
else:
    skel_gdf = gpd.read_file(str(skel_file))
    net_gdf  = gpd.read_file(str(net_file))

    utm_crs = skel_gdf.crs
    if net_gdf.crs != utm_crs:
        net_gdf = net_gdf.to_crs(utm_crs)
    print(f"Skeleton: {len(skel_gdf)} segments | Network: {len(net_gdf)} edges")

    ps_arr20, _, ps_tf20, _ = load_tif(OUT_SPEC / "path_signal.tif")
    skel_for_sample = skel_gdf.to_crs(bsi_crs) if skel_gdf.crs != bsi_crs else skel_gdf

    net_union       = net_gdf.geometry.unary_union
    MIN_DIST_M      = 15.0
    MIN_LENGTH_M    = 25.0
    MIN_SPEC_SIGNAL = 0.35

    unmod = []
    for (_, row), (_, srow) in zip(skel_gdf.iterrows(), skel_for_sample.iterrows()):
        geom        = row.geometry
        geom_utm_rs = srow.geometry
        if geom is None or geom.is_empty or geom.length < MIN_LENGTH_M:
            continue

        sp_vals = sample_raster_along_line(geom_utm_rs, ps_arr20, ps_tf20, 2.0)
        spec_sig = float(np.nanmean(sp_vals)) if not np.all(np.isnan(sp_vals)) else 0.0
        if spec_sig < MIN_SPEC_SIGNAL:
            continue

        st_vals = sample_raster_along_line(geom_utm_rs, stability, stab_tf, 2.0)
        stab_sig = float(np.nanmean(st_vals)) if not np.all(np.isnan(st_vals)) else 0.0

        d = geom.distance(net_union)
        if d > MIN_DIST_M:
            unmod.append({
                "geometry":       geom,
                "seg_length_m":   round(geom.length, 1),
                "dist_to_net_m":  round(d, 1),
                "spec_signal":    round(spec_sig, 4),
                "stability_mean": round(stab_sig, 4),
                "confidence":     "candidate",
            })

    if unmod:
        unmod_gdf = gpd.GeoDataFrame(unmod, crs=utm_crs)
        max_len = unmod_gdf["seg_length_m"].max()
        unmod_gdf["discovery_score"] = (
            0.50 * unmod_gdf["spec_signal"]
            + 0.30 * unmod_gdf["stability_mean"]
            + 0.20 * (unmod_gdf["seg_length_m"] / (max_len + 1e-9))
        ).round(4)
        unmod_gdf = unmod_gdf.sort_values("discovery_score", ascending=False).reset_index(drop=True)
        unmod_gdf.to_file(OUT_VEC / "unmodelled_corridors.geojson", driver="GeoJSON")
        print(f"Unmodelled corridor candidates: {len(unmod_gdf)} segments ({unmod_gdf['seg_length_m'].sum():.0f} m total)")
        print(unmod_gdf[["seg_length_m","dist_to_net_m","spec_signal","stability_mean","discovery_score"]].head(5).to_string(index=False))
    else:
        unmod_gdf = gpd.GeoDataFrame(columns=["geometry","seg_length_m","dist_to_net_m","spec_signal","stability_mean","discovery_score"], crs=utm_crs)
        print("No unmodelled corridors found at current thresholds.")

    with rasterio.open(OUT_SPEC / "path_signal.tif") as _src:
        _b = _src.bounds
    full_extent20 = [_b.left, _b.right, _b.bottom, _b.top]

    mnf1_bg_path20 = OUT_SPEC / "MNF1.tif"
    if mnf1_bg_path20.exists():
        with rasterio.open(mnf1_bg_path20) as _s:
            _m1 = _s.read(1).astype(np.float32)
        _p2, _p98 = np.nanpercentile(_m1[~np.isnan(_m1)], [2, 98])
        bg20 = np.clip((_m1 - _p2) / (_p98 - _p2 + 1e-9), 0, 1)
    else:
        bg20 = path_signal_f

    fig, axes20 = plt.subplots(1, 2, figsize=(20, 12))
    fig.suptitle("CELL 20 - Unmodelled Corridor Discovery", fontsize=12, fontweight="bold")

    ax_l = axes20[0]
    ax_l.imshow(bg20, cmap="gray", origin="upper", extent=full_extent20, vmin=0, vmax=1, alpha=0.55)
    ax_l.imshow(path_signal_f, cmap="hot", origin="upper", extent=full_extent20, vmin=0.3, vmax=1, alpha=0.30)
    net_gdf.plot(ax=ax_l, color="#3366cc", linewidth=1.8, zorder=3)
    if not skel_gdf.empty:
        skel_gdf.plot(ax=ax_l, color="#aaddff", linewidth=1.0, alpha=0.55, zorder=4)
    if not unmod_gdf.empty:
        unmod_gdf.plot(ax=ax_l, column="discovery_score", cmap="YlOrRd", vmin=0, vmax=1, linewidth=2.5, zorder=5)
    ax_l.set_xlim(SITE_MINX, SITE_MAXX); ax_l.set_ylim(SITE_MINY, SITE_MAXY)
    ax_l.set_title("All skeleton vs. existing network (site area)", fontsize=9); ax_l.set_aspect("equal")
    ax_l.set_xlabel("Easting (m)", fontsize=7); ax_l.set_ylabel("Northing (m)", fontsize=7)
    legend_els = [
        Line2D([0],[0], color="#3366cc", linewidth=2, label="Existing network"),
        Line2D([0],[0], color="#aaddff", linewidth=1, alpha=0.7, label="Skeleton (site)"),
        Line2D([0],[0], color="#ff6600", linewidth=2.5, label="Unmodelled candidate"),
    ]
    ax_l.legend(handles=legend_els, loc="lower right", fontsize=8)

    ax_r = axes20[1]
    ax_r.imshow(bg20, cmap="gray", origin="upper", extent=full_extent20, vmin=0, vmax=1, alpha=0.55)
    ax_r.imshow(path_signal_f, cmap="hot", origin="upper", extent=full_extent20, vmin=0.3, vmax=1, alpha=0.30)
    net_gdf.plot(ax=ax_r, color="#3366cc", linewidth=1.2, alpha=0.5, zorder=3)
    if not unmod_gdf.empty:
        unmod_gdf.plot(ax=ax_r, column="discovery_score", cmap="YlOrRd", vmin=0, vmax=1, linewidth=3.0, legend=True, legend_kwds={"label": "Discovery score", "shrink": 0.6}, zorder=5)
    ax_r.set_xlim(SITE_MINX, SITE_MAXX); ax_r.set_ylim(SITE_MINY, SITE_MAXY)
    ax_r.set_title(f"Unmodelled candidates (dist>{MIN_DIST_M}m, spec≥{MIN_SPEC_SIGNAL})", fontsize=9)
    ax_r.set_aspect("equal"); ax_r.set_xlabel("Easting (m)", fontsize=7)

    plt.tight_layout()
    plt.savefig(OUT_SPEC / "20_unmodelled_corridors.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# load main path skeleton and unmodelled corridors
# load building outlines and shortest path simulation
# load electric circuit simulation and door road network layers
# load entrance clustering and proximity tree and space syntax maps
# draw combined six panel summary picture showing skeleton across all models
# save summary picture and individual detailed panel pictures to disk
_skel_path  = OUT_VEC / "path_skeleton_lines.geojson"
_unmod_path = OUT_VEC / "unmodelled_corridors.geojson"
_skel21   = gpd.read_file(str(_skel_path)).to_crs(bsi_crs)  if _skel_path.exists()  else gpd.GeoDataFrame()
_unmod21  = gpd.read_file(str(_unmod_path)).to_crs(bsi_crs) if _unmod_path.exists() else gpd.GeoDataFrame()
_fp21     = gpd.read_file(str(OUT_VEC / "footprints_attributed.geojson")).to_crs(bsi_crs)

_xlim = (SITE_MINX, SITE_MAXX)
_ylim = (SITE_MINY, SITE_MAXY)

def _skel_legend():
    return [
        Line2D([0],[0], color="cyan",    linewidth=1.5, label="Spectral skeleton"),
        Line2D([0],[0], color="#ff2200", linewidth=2.0, label="Unmodelled corridor"),
    ]

def _apply_site_limits(ax, title, xlabel=True):
    ax.set_xlim(*_xlim); ax.set_ylim(*_ylim); ax.set_aspect("equal")
    ax.set_title(title, fontsize=9, fontweight="bold")
    if xlabel:
        ax.set_xlabel("Easting (m)", fontsize=7)
    ax.tick_params(labelsize=6)

def _overlay_skeleton(ax, lw_skel=1.5, lw_unmod=2.2):
    if not _skel21.empty:
        _skel21.plot(ax=ax, color="cyan", linewidth=lw_skel, alpha=0.85, zorder=8)
    if not _unmod21.empty:
        _unmod21.plot(ax=ax, color="#ff2200", linewidth=lw_unmod, zorder=9)

def _footprint_bg(ax, alpha=0.25):
    if not _fp21.empty:
        _fp21.plot(ax=ax, facecolor="#b0c8e8", edgecolor="#6688aa", linewidth=0.4, alpha=alpha, zorder=2)

_sp_path = OUT_VEC / "shortest_paths_from_179.geojson"
gdf_sp   = gpd.read_file(str(_sp_path)).to_crs(bsi_crs) if _sp_path.exists() else gpd.GeoDataFrame()

_ckt_tif = _base / "outputs" / "circuit_current_pairwise.tif"
if _ckt_tif.exists():
    with rasterio.open(str(_ckt_tif)) as _src:
        _ckt_arr = _src.read(1).astype(np.float32)
        _ckt_ext = [_src.bounds.left, _src.bounds.right, _src.bounds.bottom, _src.bounds.top]
        _ckt_nodata = _src.nodata
    if _ckt_nodata is not None:
        _ckt_arr[_ckt_arr == _ckt_nodata] = np.nan
    _ckt_arr = np.where(np.isnan(_ckt_arr), 0, _ckt_arr)
    _ckt_max = np.nanmax(_ckt_arr)
    if _ckt_max > 0:
        _ckt_arr = _ckt_arr / _ckt_max
else:
    _ckt_arr = np.zeros((10,10), dtype=np.float32)
    _ckt_ext = [SITE_MINX, SITE_MAXX, SITE_MINY, SITE_MAXY]

_drn_roads_path = OUT_VEC / "door_roads_network.geojson"
_drn_conn_path  = OUT_VEC / "door_roads_connections.geojson"
_drn_hubs_path  = OUT_VEC / "door_roads_intersections.geojson"

gdf_drn_roads = gpd.read_file(str(_drn_roads_path)).to_crs(bsi_crs) if _drn_roads_path.exists() else gpd.GeoDataFrame()
gdf_drn_conn  = gpd.read_file(str(_drn_conn_path)).to_crs(bsi_crs)  if _drn_conn_path.exists()  else gpd.GeoDataFrame()
gdf_drn_hubs  = gpd.read_file(str(_drn_hubs_path)).to_crs(bsi_crs)  if _drn_hubs_path.exists()  else gpd.GeoDataFrame()

_shared_dir = str((_base / "path_finding").resolve())
if _shared_dir not in sys.path:
    sys.path.insert(0, _shared_dir)
try:
    import shared
    import preprocessing
    _, _drn_doors_raw, _, _ = shared.get_base_preprocessed_data()
    gdf_drn_doors = _drn_doors_raw.to_crs(bsi_crs)
    _road_pts = []
    for _idx, _row in _drn_doors_raw.iterrows():
        _poly = _fp21.loc[_row["shp_idx"]].geometry if "shp_idx" in _row and _row["shp_idx"] in _fp21.index else None
        if _poly is None:
            _door_pt_gs = gpd.GeoSeries([_row["door_pt"]], crs=_drn_doors_raw.crs).to_crs(bsi_crs)
            _p = _door_pt_gs.iloc[0]
            _road_pts.append({"x": _p.x, "y": _p.y})
        else:
            _mx, _my, _nx, _ny, _el = preprocessing.best_wall(_poly, _row["direction"])
            _door_pt_gs = gpd.GeoSeries([_row["door_pt"]], crs=_drn_doors_raw.crs).to_crs(bsi_crs)
            _p = _door_pt_gs.iloc[0]
            _road_pts.append({"x": _p.x + _nx * 3.5, "y": _p.y + _ny * 3.5})
    df_drn_pts = pd.DataFrame(_road_pts)
except Exception as _e:
    print(f"Notice: could not load door geometry details ({_e}), plotting vector layers only.")
    gdf_drn_doors = gpd.GeoDataFrame()
    df_drn_pts = pd.DataFrame()

def _plot_door_road_network(ax, alpha_scale=1.0):
    if not gdf_drn_hubs.empty and "weight" in gdf_drn_hubs.columns:
        _sizes = np.minimum(350, 60 + 35 * gdf_drn_hubs["weight"])
        gdf_drn_hubs.plot(ax=ax, color="#8e44ad", markersize=_sizes, alpha=0.35 * alpha_scale, edgecolor="none", zorder=3)
    if not gdf_drn_roads.empty and "is_double" in gdf_drn_roads.columns:
        _single = gdf_drn_roads[~gdf_drn_roads["is_double"]]
        if not _single.empty:
            _single.plot(ax=ax, color="#3498db", linewidth=2.0, alpha=0.9 * alpha_scale, zorder=4)
        _double = gdf_drn_roads[gdf_drn_roads["is_double"]]
        if not _double.empty:
            _double.plot(ax=ax, color="#e67e22", linewidth=5.0, alpha=0.9 * alpha_scale, zorder=4)
    elif not gdf_drn_roads.empty:
        gdf_drn_roads.plot(ax=ax, color="#3498db", linewidth=2.0, alpha=0.9 * alpha_scale, zorder=4)
    if not gdf_drn_conn.empty:
        gdf_drn_conn.plot(ax=ax, color="#2980b9", linestyle="--", linewidth=1.5, alpha=0.85 * alpha_scale, zorder=4)
    if not gdf_drn_doors.empty:
        _dir_clr = {"N": "#f5a623", "S": "#7ed321", "E": "#9013fe", "W": "#d0021b", "S_fallback": "#aaaaaa", None: "#aaaaaa"}
        for _, _row in gdf_drn_doors.iterrows():
            if _row.geometry is not None and not _row.geometry.is_empty:
                _xs, _ys = _row.geometry.xy
                ax.plot(_xs, _ys, color=_dir_clr.get(_row.get("direction"), "#aaa"), linewidth=2.0, alpha=alpha_scale, zorder=5)
    if not df_drn_pts.empty and "x" in df_drn_pts.columns:
        ax.scatter(df_drn_pts["x"], df_drn_pts["y"], color="red", s=10, alpha=0.6 * alpha_scale, zorder=5)

_drn_legend_handles = [
    Line2D([0], [0], color="#3498db", lw=2, label="Single-Sided Road"),
    Line2D([0], [0], color="#e67e22", lw=5, label="Double-Sided Road (Two-Way)"),
    Line2D([0], [0], color="#2980b9", linestyle="--", lw=1.5, label="Connection Pathway"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#8e44ad", markeredgecolor="none", alpha=0.5, markersize=8, label="Congregation Hub"),
    Line2D([0], [0], marker=".", color="w", markerfacecolor="red", markersize=6, label="Door Road Points")
]

_ec_path  = OUT_VEC / "entrance_clustering_network.geojson"
_ec_hubs  = OUT_VEC / "entrance_clustering_hubs.geojson"
gdf_ec    = gpd.read_file(str(_ec_path)).to_crs(bsi_crs)  if _ec_path.exists()  else gpd.GeoDataFrame()
gdf_hubs  = gpd.read_file(str(_ec_hubs)).to_crs(bsi_crs)  if _ec_hubs.exists()  else gpd.GeoDataFrame()

_prox_path = OUT_VEC / "proximity_steiner_tree.geojson"
gdf_prox   = gpd.read_file(str(_prox_path)).to_crs(bsi_crs) if _prox_path.exists() else gpd.GeoDataFrame()

_ss_tif  = _base / "outputs" / "space_syntax_integration_subsampled.tif"
_ss_path = OUT_VEC / "space_syntax_axial_lines_subsampled.geojson"
if _ss_tif.exists():
    with rasterio.open(str(_ss_tif)) as _src:
        _ss_arr = _src.read(1).astype(np.float32)
        _ss_ext = [_src.bounds.left, _src.bounds.right, _src.bounds.bottom, _src.bounds.top]
        _ss_nd  = _src.nodata
    if _ss_nd is not None:
        _ss_arr[_ss_arr == _ss_nd] = np.nan
else:
    _ss_arr = np.zeros((10,10), dtype=np.float32)
    _ss_ext = [SITE_MINX, SITE_MAXX, SITE_MINY, SITE_MAXY]
gdf_ss = gpd.read_file(str(_ss_path)).to_crs(bsi_crs) if _ss_path.exists() else gpd.GeoDataFrame()

fig21 = plt.figure(figsize=(24, 36))
fig21.suptitle("CELL 21 — Spectral Skeleton Overlay on Path-Finding Approaches", fontsize=13, fontweight="bold", y=0.995)
gs = gridspec.GridSpec(3, 2, figure=fig21, hspace=0.08, wspace=0.05)

ax_A = fig21.add_subplot(gs[0, 0])
_footprint_bg(ax_A)
if not gdf_sp.empty:
    _col = "path_count" if "path_count" in gdf_sp.columns else None
    if _col:
        _mx = gdf_sp[_col].max()
        gdf_sp.plot(ax=ax_A, column=_col, cmap="Reds", norm=mplcolors.Normalize(vmin=0, vmax=_mx), linewidth=1.5, alpha=0.85, zorder=3)
    else:
        gdf_sp.plot(ax=ax_A, color="#cc3300", linewidth=1.5, alpha=0.85, zorder=3)
_overlay_skeleton(ax_A)
_apply_site_limits(ax_A, "A  Shortest Paths — Chapel 179")
ax_A.legend(handles=[Line2D([0],[0], color="#cc3300", lw=1.5, label="Shortest paths")] + _skel_legend(), fontsize=6, loc="lower right")
ax_A.set_ylabel("Northing (m)", fontsize=7)

ax_B = fig21.add_subplot(gs[0, 1])
_footprint_bg(ax_B)
_p2b, _p98b = np.nanpercentile(_ckt_arr[_ckt_arr > 0], [2, 98]) if np.any(_ckt_arr > 0) else (0, 1)
ax_B.imshow(_ckt_arr, cmap="YlOrRd", origin="upper", extent=_ckt_ext, vmin=_p2b, vmax=_p98b, alpha=0.80, zorder=1)
_overlay_skeleton(ax_B)
_apply_site_limits(ax_B, "B  Circuit Model — Pairwise Current Density")
ax_B.legend(handles=_skel_legend(), fontsize=6, loc="lower right")

ax_C = fig21.add_subplot(gs[1, 0])
_footprint_bg(ax_C)
_plot_door_road_network(ax_C)
_overlay_skeleton(ax_C)
_apply_site_limits(ax_C, "C  Door-Road Network")
ax_C.legend(handles=_drn_legend_handles + _skel_legend(), fontsize=6, loc="lower right")
ax_C.set_ylabel("Northing (m)", fontsize=7)

ax_D = fig21.add_subplot(gs[1, 1])
_footprint_bg(ax_D)
if not gdf_ec.empty:
    _obs  = gdf_ec[gdf_ec.get("crosses_building", False) == False] if "crosses_building" in gdf_ec.columns else gdf_ec
    _xing = gdf_ec[gdf_ec.get("crosses_building", False) == True]  if "crosses_building" in gdf_ec.columns else gpd.GeoDataFrame()
    _obs.plot(ax=ax_D, color="#22aa44", linewidth=2.0, zorder=3, alpha=0.9)
    if not _xing.empty:
        _xing.plot(ax=ax_D, color="#ee8800", linewidth=1.5, linestyle="--", zorder=3, alpha=0.75)
if not gdf_hubs.empty:
    gdf_hubs.plot(ax=ax_D, color="#aa00cc", markersize=30, zorder=5, alpha=0.85)
_overlay_skeleton(ax_D)
_apply_site_limits(ax_D, "D  Entrance Clustering — Hub Network")
ax_D.legend(handles=[Line2D([0],[0], color="#22aa44", lw=2, label="Hub edge (clear)"), Line2D([0],[0], color="#ee8800", lw=1.5, ls="--", label="Hub edge (crossing)"), Line2D([0],[0], marker="o", color="#aa00cc", lw=0, markersize=6, label="Access hub")] + _skel_legend(), fontsize=6, loc="lower right")

ax_E = fig21.add_subplot(gs[2, 0])
_footprint_bg(ax_E)
if not gdf_prox.empty:
    gdf_prox.plot(ax=ax_E, color="#884400", linewidth=2.5, alpha=0.85, zorder=3)
_overlay_skeleton(ax_E)
_apply_site_limits(ax_E, "E  Proximity — Steiner Tree")
ax_E.legend(handles=[Line2D([0],[0], color="#884400", lw=2.5, label="Steiner tree")] + _skel_legend(), fontsize=6, loc="lower right")
ax_E.set_ylabel("Northing (m)", fontsize=7)

ax_F = fig21.add_subplot(gs[2, 1])
_footprint_bg(ax_F)
_p2f = np.nanpercentile(_ss_arr[~np.isnan(_ss_arr)], 2)  if not np.all(np.isnan(_ss_arr)) else 0
_p98f = np.nanpercentile(_ss_arr[~np.isnan(_ss_arr)], 98) if not np.all(np.isnan(_ss_arr)) else 1
ax_F.imshow(_ss_arr, cmap="RdYlGn", origin="upper", extent=_ss_ext, vmin=_p2f, vmax=_p98f, alpha=0.70, zorder=1)
if not gdf_ss.empty and "integration_mean_norm" in gdf_ss.columns:
    gdf_ss.plot(ax=ax_F, column="integration_mean_norm", cmap="RdYlGn", linewidth=1.0, alpha=0.55, zorder=3, legend=False)
elif not gdf_ss.empty:
    gdf_ss.plot(ax=ax_F, color="#336600", linewidth=0.8, alpha=0.50, zorder=3)
_overlay_skeleton(ax_F)
_apply_site_limits(ax_F, "F  Space Syntax — Axial Integration")
ax_F.legend(handles=_skel_legend(), fontsize=6, loc="lower right")

fig21.savefig(OUT_SPEC / "21_skeleton_on_path_approaches.png", dpi=150, bbox_inches="tight")
plt.close(fig21)

panels_22 = [
    ("A", "Shortest Paths — Chapel 179",     ax_A),
    ("B", "Circuit Pairwise Current Density", ax_B),
    ("C", "Door-Road Network",                ax_C),
    ("D", "Entrance Clustering Hub Network",  ax_D),
    ("E", "Proximity Steiner Tree",           ax_E),
    ("F", "Space Syntax Axial Integration",   ax_F),
]
_indiv_dir = OUT_SPEC / "21_individual_panels"
_indiv_dir.mkdir(parents=True, exist_ok=True)

for _tag, _label, _src_ax in panels_22:
    _fig_i, _ax_i = plt.subplots(figsize=(11, 15))
    _ax_i.set_title(f"CELL 21{_tag}  {_label}\n(cyan = spectral skeleton | red = unmodelled corridors)", fontsize=10, fontweight="bold")
    if _tag == "A":
        _footprint_bg(_ax_i)
        if not gdf_sp.empty:
            gdf_sp.plot(ax=_ax_i, color="#cc3300", linewidth=1.8, alpha=0.85, zorder=3)
    elif _tag == "B":
        _footprint_bg(_ax_i)
        _ax_i.imshow(_ckt_arr, cmap="YlOrRd", origin="upper", extent=_ckt_ext, vmin=_p2b, vmax=_p98b, alpha=0.80, zorder=1)
    elif _tag == "C":
        _footprint_bg(_ax_i)
        _plot_door_road_network(_ax_i)
    elif _tag == "D":
        _footprint_bg(_ax_i)
        if not gdf_ec.empty:
            _obs_i  = gdf_ec[gdf_ec.get("crosses_building", False) == False] if "crosses_building" in gdf_ec.columns else gdf_ec
            _xing_i = gdf_ec[gdf_ec.get("crosses_building", False) == True]  if "crosses_building" in gdf_ec.columns else gpd.GeoDataFrame()
            _obs_i.plot(ax=_ax_i, color="#22aa44", linewidth=2.0, alpha=0.90, zorder=3)
            if not _xing_i.empty:
                _xing_i.plot(ax=_ax_i, color="#ee8800", linewidth=1.5, linestyle="--", alpha=0.80, zorder=3)
        if not gdf_hubs.empty:
            gdf_hubs.plot(ax=_ax_i, color="#aa00cc", markersize=40, zorder=5, alpha=0.85)
    elif _tag == "E":
        _footprint_bg(_ax_i)
        if not gdf_prox.empty:
            gdf_prox.plot(ax=_ax_i, color="#884400", linewidth=3.0, alpha=0.85, zorder=3)
    elif _tag == "F":
        _footprint_bg(_ax_i)
        _ax_i.imshow(_ss_arr, cmap="RdYlGn", origin="upper", extent=_ss_ext, vmin=_p2f, vmax=_p98f, alpha=0.70, zorder=1)
        if not gdf_ss.empty:
            gdf_ss.plot(ax=_ax_i, color="#336600", linewidth=0.8, alpha=0.50, zorder=3)

    _overlay_skeleton(_ax_i)
    _apply_site_limits(_ax_i, f"21{_tag}  {_label}")
    _ax_i.set_ylabel("Northing (m)", fontsize=7)
    _ax_i.legend(handles=_drn_legend_handles + _skel_legend() if _tag == "C" else _skel_legend(), fontsize=7, loc="lower right")
    _out_name = _indiv_dir / f"21{_tag}_{_label.replace(' ', '_').replace('—', '-')}.png"
    _fig_i.savefig(_out_name, dpi=150, bbox_inches="tight")
    plt.close(_fig_i)
print("CELL 21-22 done.")



In [ ]:
# define functions to check if skeleton lines agree with each model
# check overlap between skeleton and shortest path lines
# check overlap against electric flow density map
# check overlap against door road network
# check overlap against entrance hubs and proximity tree and space syntax
# draw and save six panel agreement map highlighting matching paths in gold
AGREE_DIST_M        = 12.0
AGREE_RASTER_THRESH = 0.60
GOLD = "#FFD700"
CYAN = "cyan"
RED  = "#ff2200"

def _split_skeleton_by_overlap_vector(skel_gdf, network_gdf, dist_m=AGREE_DIST_M):
    if network_gdf.empty or skel_gdf.empty:
        return gpd.GeoDataFrame(), skel_gdf
    net_union = network_gdf.geometry.unary_union.buffer(dist_m)
    agree_mask = skel_gdf.geometry.apply(lambda g: g.intersects(net_union))
    return skel_gdf[agree_mask], skel_gdf[~agree_mask]

def _split_skeleton_by_overlap_raster(skel_gdf, raster_arr, raster_ext, thresh=AGREE_RASTER_THRESH):
    if skel_gdf.empty:
        return gpd.GeoDataFrame(), skel_gdf
    left, right, bottom, top = raster_ext
    nrows, ncols = raster_arr.shape
    px_w = (right - left) / ncols
    px_h = (top  - bottom) / nrows
    r_max = np.nanmax(raster_arr)
    threshold_val = thresh * r_max if r_max > 0 else 0

    def _val_at(geom):
        pt = geom.centroid
        col = int((pt.x - left) / px_w)
        row = int((top  - pt.y) / px_h)
        if 0 <= row < nrows and 0 <= col < ncols:
            return raster_arr[row, col]
        return 0.0

    vals = skel_gdf.geometry.apply(_val_at)
    agree_mask = vals >= threshold_val
    return skel_gdf[agree_mask], skel_gdf[~agree_mask]

def _draw_overlap_legend(ax, approach_color, approach_label, extra_handles=None):
    handles = []
    if extra_handles:
        handles.extend(extra_handles)
    elif approach_color and approach_label:
        handles.append(Line2D([0],[0], color=approach_color, lw=2, label=approach_label))
    handles.extend([
        Line2D([0],[0], color=GOLD,           lw=2.5, label="Skeleton Γ£ô (agrees)"),
        Line2D([0],[0], color=CYAN,           lw=1.5, label="Skeleton Γ£ù (no match)"),
        Line2D([0],[0], color=RED,            lw=2,   label="Unmodelled corridor"),
    ])
    ax.legend(handles=handles, fontsize=6, loc="lower right")

def _overlay_split_skeleton(ax, agree, disagree):
    if not disagree.empty:
        disagree.plot(ax=ax, color=CYAN, linewidth=1.5, alpha=0.75, zorder=8)
    if not agree.empty:
        agree.plot(ax=ax, color=GOLD, linewidth=2.8, alpha=1.00, zorder=9)
    if not _unmod21.empty:
        _unmod21.plot(ax=ax, color=RED, linewidth=2.2, zorder=10)

_ag_A, _dis_A = _split_skeleton_by_overlap_vector(_skel21, gdf_sp)
_ag_B, _dis_B = _split_skeleton_by_overlap_raster(_skel21, _ckt_arr, _ckt_ext)

gdf_drn_geom = gpd.GeoDataFrame(
    pd.concat([gdf_drn_roads[["geometry"]], gdf_drn_conn[["geometry"]]], ignore_index=True),
    crs=bsi_crs
) if not gdf_drn_roads.empty or not gdf_drn_conn.empty else gpd.GeoDataFrame()
_ag_C, _dis_C = _split_skeleton_by_overlap_vector(_skel21, gdf_drn_geom)

_ag_D, _dis_D = _split_skeleton_by_overlap_vector(_skel21, gdf_ec)
_ag_E, _dis_E = _split_skeleton_by_overlap_vector(_skel21, gdf_prox)
_ag_F, _dis_F = _split_skeleton_by_overlap_raster(_skel21, _ss_arr, _ss_ext)

for tag, ag in [("A", _ag_A), ("B", _ag_B), ("C", _ag_C), ("D", _ag_D), ("E", _ag_E), ("F", _ag_F)]:
    total = len(_skel21)
    pct   = 100 * len(ag) / total if total else 0
    print(f"Panel {tag}: {len(ag)}/{total} skeleton segments agree ({pct:.0f}%)")

fig23 = plt.figure(figsize=(24, 36))
fig23.suptitle("CELL 23 — Skeleton Agreement with Path-Finding Approaches", fontsize=13, fontweight="bold", y=0.995)
gs23 = gridspec.GridSpec(3, 2, figure=fig23, hspace=0.08, wspace=0.05)

ax23_A = fig23.add_subplot(gs23[0, 0])
_footprint_bg(ax23_A)
if not gdf_sp.empty:
    gdf_sp.plot(ax=ax23_A, color="#cc3300", linewidth=1.2, alpha=0.55, zorder=3)
_overlay_split_skeleton(ax23_A, _ag_A, _dis_A)
_apply_site_limits(ax23_A, "A  Shortest Paths — Agreement")
_draw_overlap_legend(ax23_A, "#cc3300", "Shortest paths")
ax23_A.set_ylabel("Northing (m)", fontsize=7)

ax23_B = fig23.add_subplot(gs23[0, 1])
_footprint_bg(ax23_B)
ax23_B.imshow(_ckt_arr, cmap="YlOrRd", origin="upper", extent=_ckt_ext, vmin=_p2b, vmax=_p98b, alpha=0.65, zorder=1)
_overlay_split_skeleton(ax23_B, _ag_B, _dis_B)
_apply_site_limits(ax23_B, "B  Circuit Density — Agreement")
_draw_overlap_legend(ax23_B, "#cc6600", "High current zone")

ax23_C = fig23.add_subplot(gs23[1, 0])
_footprint_bg(ax23_C)
_plot_door_road_network(ax23_C, alpha_scale=0.65)
_overlay_split_skeleton(ax23_C, _ag_C, _dis_C)
_apply_site_limits(ax23_C, "C  Door-Road Network — Agreement")
_draw_overlap_legend(ax23_C, None, None, extra_handles=_drn_legend_handles)
ax23_C.set_ylabel("Northing (m)", fontsize=7)

ax23_D = fig23.add_subplot(gs23[1, 1])
_footprint_bg(ax23_D)
if not gdf_ec.empty:
    _obs23  = gdf_ec[gdf_ec.get("crosses_building", False) == False] if "crosses_building" in gdf_ec.columns else gdf_ec
    _xing23 = gdf_ec[gdf_ec.get("crosses_building", False) == True]  if "crosses_building" in gdf_ec.columns else gpd.GeoDataFrame()
    _obs23.plot(ax=ax23_D, color="#22aa44", linewidth=1.5, alpha=0.55, zorder=3)
    if not _xing23.empty:
        _xing23.plot(ax=ax23_D, color="#ee8800", linewidth=1.0, linestyle="--", alpha=0.55, zorder=3)
if not gdf_hubs.empty:
    gdf_hubs.plot(ax=ax23_D, color="#aa00cc", markersize=20, alpha=0.60, zorder=5)
_overlay_split_skeleton(ax23_D, _ag_D, _dis_D)
_apply_site_limits(ax23_D, "D  Entrance Clustering — Agreement")
_draw_overlap_legend(ax23_D, "#22aa44", "Hub network")

ax23_E = fig23.add_subplot(gs23[2, 0])
_footprint_bg(ax23_E)
if not gdf_prox.empty:
    gdf_prox.plot(ax=ax23_E, color="#884400", linewidth=2.5, alpha=0.55, zorder=3)
_overlay_split_skeleton(ax23_E, _ag_E, _dis_E)
_apply_site_limits(ax23_E, "E  Proximity Steiner — Agreement")
_draw_overlap_legend(ax23_E, "#884400", "Steiner tree")
ax23_E.set_ylabel("Northing (m)", fontsize=7)

ax23_F = fig23.add_subplot(gs23[2, 1])
_footprint_bg(ax23_F)
ax23_F.imshow(_ss_arr, cmap="RdYlGn", origin="upper", extent=_ss_ext, vmin=_p2f, vmax=_p98f, alpha=0.60, zorder=1)
if not gdf_ss.empty:
    gdf_ss.plot(ax=ax23_F, color="#336600", linewidth=0.7, alpha=0.45, zorder=3)
_overlay_split_skeleton(ax23_F, _ag_F, _dis_F)
_apply_site_limits(ax23_F, "F  Space Syntax — Agreement")
_draw_overlap_legend(ax23_F, "#336600", "High integration zone")

fig23.savefig(OUT_SPEC / "23_skeleton_agreement.png", dpi=150, bbox_inches="tight")
plt.close(fig23)

_indiv_dir23 = OUT_SPEC / "23_individual_panels"
_indiv_dir23.mkdir(parents=True, exist_ok=True)

_panels23 = [
    ("A", "Shortest_Paths",        _ag_A, _dis_A, "#cc3300", "Shortest paths",
     lambda ax: gdf_sp.plot(ax=ax, color="#cc3300", linewidth=1.2, alpha=0.55, zorder=3) if not gdf_sp.empty else None),
    ("B", "Circuit_Density",       _ag_B, _dis_B, "#cc6600", "High current zone",
     lambda ax: ax.imshow(_ckt_arr, cmap="YlOrRd", origin="upper", extent=_ckt_ext, vmin=_p2b, vmax=_p98b, alpha=0.65, zorder=1)),
    ("C", "Door_Road_Network",     _ag_C, _dis_C, None, None,
     lambda ax: _plot_door_road_network(ax, alpha_scale=0.65)),
    ("D", "Entrance_Clustering",   _ag_D, _dis_D, "#22aa44", "Hub network",
     lambda ax: [_obs23.plot(ax=ax, color="#22aa44", linewidth=1.5, alpha=0.55, zorder=3),
                 gdf_hubs.plot(ax=ax, color="#aa00cc", markersize=20, alpha=0.60, zorder=5)] if not gdf_ec.empty else None),
    ("E", "Proximity_Steiner",     _ag_E, _dis_E, "#884400", "Steiner tree",
     lambda ax: gdf_prox.plot(ax=ax, color="#884400", linewidth=2.5, alpha=0.55, zorder=3) if not gdf_prox.empty else None),
    ("F", "Space_Syntax",          _ag_F, _dis_F, "#336600", "High integration zone",
     lambda ax: [ax.imshow(_ss_arr, cmap="RdYlGn", origin="upper", extent=_ss_ext, vmin=_p2f, vmax=_p98f, alpha=0.60, zorder=1),
                 gdf_ss.plot(ax=ax, color="#336600", linewidth=0.7, alpha=0.45, zorder=3) if not gdf_ss.empty else None]),
]

for _tag23, _name23, _ag, _dis, _col23, _lbl23, _bg_fn in _panels23:
    _fig_i23, _ax_i23 = plt.subplots(figsize=(11, 15))
    _ax_i23.set_title(f"23{_tag23}  {_name23.replace('_',' ')}\nGOLD = skeleton agrees | CYAN = no match | RED = unmodelled", fontsize=10, fontweight="bold")
    _footprint_bg(_ax_i23)
    _bg_fn(_ax_i23)
    _overlay_split_skeleton(_ax_i23, _ag, _dis)
    _apply_site_limits(_ax_i23, f"23{_tag23}  {_name23.replace('_',' ')}")
    _ax_i23.set_ylabel("Northing (m)", fontsize=7)
    if _tag23 == "C":
        _draw_overlap_legend(_ax_i23, None, None, extra_handles=_drn_legend_handles)
    else:
        _draw_overlap_legend(_ax_i23, _col23, _lbl23)
    _fig_i23.savefig(_indiv_dir23 / f"23{_tag23}_{_name23}.png", dpi=150, bbox_inches="tight")
    plt.close(_fig_i23)
print("CELL 23 done.")
